In [1]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

import json
from pathlib import Path
import cv2
import numpy as np
import tqdm

In [2]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [3]:
'gpu is available' if device == 'cuda' else 'WARNING: no gpu detected working on cpu might be way slower'

'gpu is available'

In [4]:
class KeypointDataset(Dataset):
    def __init__(self, img_dir, data_file):
        self.img_dir = img_dir
        with open(data_file, 'r') as file:
            self.data_file = json.load(file)
        
        self.transforms = transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize((224,224)),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229,0.224,0.225]
            )
        ])
    
    def __len__(self):
        return len(self.data_file)
    
    def __getitem__(self, idx):
        item = self.data_file[idx]
        img = cv2.imread(Path(f"{self.img_dir}/{item['id']}.png"))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]
        img = self.transforms(img)

        kps = item['kps']
        kps = np.array(kps).flatten()
        kps = kps.astype(np.float32)
        kps[::2] *= 224.0 / w #adjust x to resized image
        kps[1::2] *= 224.0 / h #adjust y to resized image

        return img, kps





In [5]:
train_data = KeypointDataset('../datasets/tennis_court_data/images', '../datasets/tennis_court_data/data_train.json')
val_data = KeypointDataset('../datasets/tennis_court_data/images', '../datasets/tennis_court_data/data_val.json')

train_loader = DataLoader(train_data, batch_size=8, shuffle=True)
val_loader = DataLoader(val_data, batch_size=8, shuffle=True)

In [6]:
model = models.resnet50()
model.fc = torch.nn.Linear(model.fc.in_features, 14*2) # 14 points with x,y so 14 * 2
model.to(device)

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 

In [7]:
loss_fn = torch.nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters())

In [8]:
epochs = 30
train_losses = []
val_losses = []

for epoch in range(epochs):
    model.train()
    train_bar = tqdm.tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]")

    epoch_train_loss = 0.0
    for img, kps in train_bar:
        img = img.to(device)
        kps = kps.to(device)

        y_hat = model(img)
        train_loss = loss_fn(y_hat, kps)
        loss_value = train_loss.item()
        epoch_train_loss += loss_value

        optimizer.zero_grad()
        train_loss.backward()
        optimizer.step()

        train_bar.set_postfix(loss=loss_value)

    avg_train_loss = epoch_train_loss / len(train_loader)
    train_losses.append(avg_train_loss)

    model.eval()
    val_bar = tqdm.tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} [Val]")

    epoch_val_loss = 0.0
    with torch.no_grad():
        for img, kps in val_bar:
            img = img.to(device)
            kps = kps.to(device)

            y_hat = model(img)
            val_loss = loss_fn(y_hat, kps)
            loss_value = val_loss.item()
            epoch_val_loss += loss_value

            val_bar.set_postfix(loss=loss_value)

    avg_val_loss = epoch_val_loss / len(val_loader)
    val_losses.append(avg_val_loss)

    print(f"{epoch=} -- train_loss={avg_train_loss:.4f} -- val_loss={avg_val_loss:.4f}")


Epoch 1/30 [Val]: 100%|██████████| 277/277 [01:00<00:00,  4.59it/s, loss=192] 


epoch=0 -- train_loss=309.3196 -- val_loss=60.4888


Epoch 2/30 [Val]: 100%|██████████| 277/277 [00:59<00:00,  4.67it/s, loss=14.4]


epoch=1 -- train_loss=46.0778 -- val_loss=37.5962


Epoch 3/30 [Val]: 100%|██████████| 277/277 [00:57<00:00,  4.85it/s, loss=13.6]


...

epoch=27 -- train_loss=4.0714 -- val_loss=5.2146


Epoch 29/30 [Val]: 100%|██████████| 277/277 [00:59<00:00,  4.69it/s, loss=1.09] 


epoch=28 -- train_loss=2.7557 -- val_loss=4.5238


Epoch 30/30 [Val]: 100%|██████████| 277/277 [00:59<00:00,  4.68it/s, loss=0.386]

epoch=29 -- train_loss=2.1635 -- val_loss=3.8163


In [9]:
# Save entire model
import os
os.makedirs('models', exist_ok=True)
torch.save(model, "models/model_final.pth")

# Or better: save only the model state_dict (recommended)
torch.save(model.state_dict(), "models/model_final_state_dict.pth")